In [9]:
%whos function

Variable                          Type        Data/Info
-------------------------------------------------------
pca_biplot                        function    <function pca_biplot at 0x00000223DAD239C0>
pca_components                    function    <function pca_components at 0x00000223DACC02C0>
pca_correlation_heatmap           function    <function pca_correlation<...>ap at 0x00000223DAD907C0>
pca_explained_variance_plot       function    <function pca_explained_v<...>ot at 0x00000223DAD231A0>
pca_feature_importance            function    <function pca_feature_imp<...>ce at 0x00000223DAD902C0>
pca_feature_importance_weighted   function    <function pca_feature_imp<...>ed at 0x00000223DAD90180>
pca_scree_plot                    function    <function pca_scree_plot at 0x00000223DAD23CE0>


In [2]:
def pca_components(data, n_components=2, components_to_plot=[1, 2], vector_colors=None, quadrant_colors=None, helper_lines=False, figsize=(8,8)):
    """
    Creates a PCA factor loadings plot with optional vector coloring, quadrant shading, helper lines, 
    and interpretation support.

    Parameters:
    - data: DataFrame with input data for PCA (samples in rows, features in columns)
    - n_components: Number of principal components to compute (default is 2)
    - components_to_plot: List of two component indices to plot (default is [1, 2])
    - vector_colors: List of colors for the vectors (optional)
    - quadrant_colors: List of 4 colors for plot quadrants (optional)
    - helper_lines: Bool, if True, draws dashed helper lines from vector tips to the axes

    Returns:
    - DataFrame with factor loadings for all components
    """

    # Perform PCA
    pca = PCA(n_components=n_components)
    pca.fit(data)

    # Extract factor loadings
    loadings = pd.DataFrame(
        pca.components_.T,
        index=data.columns,
        columns=[f"PC{i+1}" for i in range(n_components)]
    )

    # Check validity of chosen components
    if max(components_to_plot) > n_components or len(components_to_plot) != 2:
        raise ValueError(f"components_to_plot must contain exactly two indices <= {n_components}.")

    # Select components to plot
    pc_x, pc_y = f"PC{components_to_plot[0]}", f"PC{components_to_plot[1]}"
    loadings_subset = loadings[[pc_x, pc_y]]

    # Default vector colors if not provided
    default_vector_colors = ['r', 'g', 'b', 'orange', 'purple']
    vector_colors = vector_colors if vector_colors else default_vector_colors

    # Default quadrant colors if not provided
    default_quadrants = ['lightblue', 'lightyellow', 'lightgreen', 'lightpink']
    quadrant_colors = quadrant_colors if quadrant_colors else default_quadrants

    # Create the plot
    plt.figure(figsize=figsize)

    # Draw unit circle
    unit_circle = plt.Circle((0, 0), 1, color='blue', fill=False, linestyle="--", linewidth=1.5, label="Unit circle")
    plt.gca().add_artist(unit_circle)

    # Shade quadrants if colors are provided
    plt.axhspan(0, 1.0, facecolor=quadrant_colors[0], alpha=0.3, label=f"{pc_x}+ / {pc_y}+")
    plt.axhspan(-1.0, 0, facecolor=quadrant_colors[1], alpha=0.3, label=f"{pc_x}+ / {pc_y}-")
    plt.axvspan(-1.0, 0, facecolor=quadrant_colors[2], alpha=0.3, label=f"{pc_x}- / {pc_y}-")
    plt.axvspan(0, 1.0, facecolor=quadrant_colors[3], alpha=0.3, label=f"{pc_x}- / {pc_y}+")

    # Draw factor loading vectors
    for i, feature in enumerate(loadings_subset.index):
        x, y = loadings_subset.loc[feature, pc_x], loadings_subset.loc[feature, pc_y]
        correlation = np.sqrt(x**2 + y**2)

        # Draw arrow
        plt.arrow(0, 0, x, y, color=vector_colors[i % len(vector_colors)], alpha=0.9, head_width=0.04, linewidth=1.5)

        # Add feature labels
        plt.text(x * 1.3, y * 1.3, f"{feature}\n({correlation:.2f})", color=vector_colors[i % len(vector_colors)],
                 fontsize=10, ha='center', va='center')

        # Draw helper dashed lines if requested
        if helper_lines:
            plt.plot([x, x], [y, 0], linestyle="--", color='gray', alpha=0.7)
            plt.plot([x, 0], [y, y], linestyle="--", color='gray', alpha=0.7)

    plt.xlim(-1.05, 1.05)
    plt.ylim(-1.05, 1.05)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.title(f"PCA factor loadings plot ({pc_x} vs {pc_y})", fontsize=16)
    plt.xlabel(pc_x, fontsize=14)
    plt.ylabel(pc_y, fontsize=14)
    plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)
    plt.legend(loc="upper left", fontsize=10, frameon=True, shadow=True)

    plt.show()
    return loadings

In [10]:
def pca_biplot(data, n_components=2, components_to_plot=[1, 2], vector_colors=None, point_color=None, helper_lines=False, figsize=(8,8)):
    """
    Creates a PCA biplot showing both observations and factor loadings (vectors).

    Parameters:
    - data: DataFrame with input data for PCA (samples in rows, features in columns)
    - n_components: Number of principal components to compute (default is 2)
    - components_to_plot: List of two component indices to display on the plot (default [1, 2])
    - vector_colors: List of colors for the feature vectors (optional)
    - point_color: Color for sample points (optional)
    - helper_lines: Bool, if True, draws dashed helper lines from vector tips to the axes

    Returns:
    - DataFrame with factor loadings for all components
    - DataFrame with projected observations onto the selected components
    """

    # Perform PCA
    pca = PCA(n_components=n_components)
    pca_results = pca.fit_transform(data)

    # Extract factor loadings
    loadings = pd.DataFrame(
        pca.components_.T,
        index=data.columns,
        columns=[f"PC{i+1}" for i in range(n_components)]
    )

    # Validate selected components
    if max(components_to_plot) > n_components or len(components_to_plot) != 2:
        raise ValueError(f"components_to_plot must contain exactly two indices <= {n_components}.")

    # Get components for plotting
    pc_x, pc_y = f"PC{components_to_plot[0]}", f"PC{components_to_plot[1]}"
    loadings_subset = loadings[[pc_x, pc_y]]

    # Project observations onto the selected components
    projections = pd.DataFrame(pca_results, columns=[f"PC{i+1}" for i in range(n_components)])

    # Default vector and point colors
    default_vector_colors = ['r', 'g', 'b', 'orange', 'purple']
    vector_colors = vector_colors if vector_colors else default_vector_colors

    default_point_color = 'blue'
    point_color = point_color if point_color else default_point_color

    # Create the plot
    plt.figure(figsize=figsize)

    # Plot observations as points
    plt.scatter(
        projections[pc_x],
        projections[pc_y],
        color=point_color,
        alpha=0.6,
        edgecolors="k",
        label="Observations"
    )

    # Plot feature vectors
    for i, feature in enumerate(loadings_subset.index):
        x, y = loadings_subset.loc[feature, pc_x], loadings_subset.loc[feature, pc_y]

        # Draw the feature vector
        plt.arrow(0, 0, x, y, color=vector_colors[i % len(vector_colors)], alpha=0.9, head_width=0.04, linewidth=1.5)

        # Add feature name
        plt.text(x * 1.2, y * 1.2, feature, color=vector_colors[i % len(vector_colors)], fontsize=12, ha='center', va='center')

        # Draw helper dashed lines if requested
        if helper_lines:
            plt.plot([x, x], [y, 0], linestyle="--", color='gray', alpha=0.7)
            plt.plot([x, 0], [y, y], linestyle="--", color='gray', alpha=0.7)

    # Draw axis lines
    plt.axhline(0, color='black', linewidth=0.8, linestyle="--")
    plt.axvline(0, color='black', linewidth=0.8, linestyle="--")

    # Set equal aspect ratio
    plt.gca().set_aspect('equal', adjustable='box')

    # Additional plot settings
    plt.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)
    plt.title(f"PCA Biplot ({pc_x} vs {pc_y})", fontsize=14)
    plt.xlabel(pc_x, fontsize=14)
    plt.ylabel(pc_y, fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.legend(loc="upper left", fontsize=12, frameon=True, shadow=True)
    plt.show()

    return loadings, projections

In [4]:
def pca_explained_variance_plot(data, max_components=10, png_file_name="explained_variance_plot_pca.png", figsize=(8,6)):
    """
    Creates a plot showing the percentage of explained variance for each principal component in PCA.

    Parameters:
    - data: DataFrame with input data for PCA (samples in rows, features in columns)
    - max_components: Maximum number of components to analyze (default: 10)

    Returns:
    - DataFrame with explained variance values for each component
    """

    # Perform PCA for the maximum specified number of components
    pca = PCA(n_components=min(max_components, data.shape[1]))
    pca.fit(data)

    # Extract explained variance
    explained_variance = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance)

    # Create Explained Variance Plot
    plt.figure(figsize=figsize)
    plt.bar(range(1, len(explained_variance) + 1), explained_variance, alpha=0.7, label="Individual Variance")
    plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='--', color='red', label="Cumulative Variance")

    # Reference lines
    plt.axhline(0.8, color='gray', linestyle="--", alpha=0.7, label="80% Explained Variance")
    plt.axhline(0.9, color='gray', linestyle="--", alpha=0.7, label="90% Explained Variance")

    plt.xlabel("Number of Principal Components", fontsize=12)
    plt.ylabel("Explained Variance Ratio", fontsize=12)
    plt.title("Explained Variance Plot of PCA", fontsize=14)
    plt.xticks(range(1, len(explained_variance) + 1))
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    plt.savefig(png_file_name, dpi=300, bbox_inches='tight', pad_inches=0.1)
    
    plt.show()

    # Return results in a DataFrame
    results_df = pd.DataFrame({
        "Component": range(1, len(explained_variance) + 1),
        "Explained Variance": explained_variance,
        "Cumulative Variance": cumulative_variance
    })

    return results_df

In [5]:
def pca_scree_plot(data, max_components=10, png_file_name="scree_plot_pca.png", figsize=(8,6)):
    """
    Creates a Scree Plot for PCA, showing the rate of decline in eigenvalues 
    (percentage of explained variance) for each principal component.

    Parameters:
    - data: DataFrame with input data for PCA (samples in rows, features in columns)
    - max_components: Maximum number of components to analyze (default: 10)

    Returns:
    - DataFrame with eigenvalues and percentage of explained variance for each component
    """

    # Perform PCA for the specified number of components
    pca = PCA(n_components=min(max_components, data.shape[1]))
    pca.fit(data)

    # Extract explained variance ratio
    explained_variance = pca.explained_variance_ratio_

    # Create Scree Plot
    plt.figure(figsize=figsize)
    plt.plot(range(1, len(explained_variance) + 1), explained_variance, marker='o', linestyle='-', color='b', label="Explained variance")
    plt.xticks(range(1, len(explained_variance) + 1))

    # Identify the "elbow" point where the drop in variance slows down
    elbow_index = np.argmax(np.diff(explained_variance)) + 1  # Index of the component after the steepest drop
    plt.axvline(elbow_index + 1, color='red', linestyle="--", label=f"Elbow criterion: PC{elbow_index + 1}")

    # Labels and title
    plt.xlabel("Number of Principal Components", fontsize=12)
    plt.ylabel("Explained Variance Ratio", fontsize=12)
    plt.title("Scree Plot of PCA", fontsize=14)
    plt.legend()
    plt.grid(axis='both', linestyle='--', alpha=0.7)

    plt.savefig(png_file_name, dpi=300, bbox_inches='tight', pad_inches=0.1)
    
    plt.show()

    # Return results in a DataFrame
    results_df = pd.DataFrame({
        "Component": range(1, len(explained_variance) + 1),
        "Explained Variance": explained_variance
    })

    return results_df

In [6]:
def pca_feature_importance_weighted(data, num_components=2, threshold=0.5, png_file_name="feature_importance_pca_weighted.png", figsize=(8,6)):
    """
    Creates a weighted Feature Importance Plot for PCA, showing the contribution of each original feature
    to the selected principal components, weighted by the explained variance ratio.
    Features exceeding the threshold are highlighted.

    Parameters:
    - data: DataFrame with input data for PCA (samples in rows, features in columns)
    - num_components: Number of principal components to analyze (default: 2)
    - threshold: Threshold value for highlighting important features (default: 0.5)
    - png_file_name: Name of the PNG file to save the plot

    Returns:
    - DataFrame with weighted feature contributions
    """

    # Perform PCA
    pca = PCA(n_components=min(num_components, data.shape[1]))
    pca.fit(data)

    # Extract loadings
    loadings = pd.DataFrame(
        pca.components_.T,
        index=data.columns,
        columns=[f"PC{i+1}" for i in range(num_components)]
    )

    # Get explained variance ratios for selected components
    explained_var = pca.explained_variance_ratio_[:num_components]

    # Calculate weighted importance score
    weighted_importance = (loadings.abs() * explained_var).sum(axis=1).sort_values(ascending=False)

    # Define colors for highlighting features above threshold
    colors = ["red" if value > threshold else "royalblue" for value in weighted_importance]

    # Create the plot
    plt.figure(figsize=figsize)
    weighted_importance.plot(kind="bar", color=colors, alpha=0.7)
    plt.axhline(threshold, color="gray", linestyle="--", label=f"Threshold = {threshold}")
    plt.xlabel("Features", fontsize=12)
    plt.ylabel("Weighted Importance Score", fontsize=12)
    plt.title("Weighted Feature Importance in PCA", fontsize=14)
    plt.xticks(rotation=45, ha="right")
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.7)

    plt.savefig(png_file_name, dpi=300, bbox_inches='tight', pad_inches=0.1)
    plt.show()

    # Return results in a DataFrame
    importance_df = pd.DataFrame({
        "Feature": weighted_importance.index,
        "Weighted Importance Score": weighted_importance.values
    })

    return importance_df


In [7]:
def pca_feature_importance(data, num_components=2, threshold=1.0, png_file_name="feature_importance_pca.png", figsize=(8,6)):
    """
    Creates a Feature Importance Plot for PCA, showing the contribution of each original feature
    to the selected principal components. Features exceeding the threshold are highlighted.

    Parameters:
    - data: DataFrame with input data for PCA (samples in rows, features in columns)
    - num_components: Number of principal components to analyze (default: 2)
    - threshold: Threshold value for highlighting important features (default: 1.0)

    Returns:
    - DataFrame with feature contributions to each selected principal component
    """

    # Perform PCA
    pca = PCA(n_components=min(num_components, data.shape[1]))
    pca.fit(data)

    # Extract loadings (weights of each feature in the principal components)
    loadings = pd.DataFrame(pca.components_.T, index=data.columns, columns=[f"PC{i+1}" for i in range(num_components)])

    # Compute absolute values for feature importance
    feature_importance = loadings.abs().sum(axis=1).sort_values(ascending=False)

    # Define colors: highlight features above threshold
    colors = ["red" if value > threshold else "royalblue" for value in feature_importance]

    # Create Feature Importance Plot
    plt.figure(figsize=figsize)
    feature_importance.plot(kind="bar", color=colors, alpha=0.7)
    plt.axhline(threshold, color="gray", linestyle="--", label=f"Threshold = {threshold}")
    plt.xlabel("Features", fontsize=12)
    plt.ylabel("Importance Score", fontsize=12)
    plt.title("Feature Importance in PCA", fontsize=14)
    plt.xticks(rotation=45, ha="right")
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.7)

   
    plt.savefig(png_file_name, dpi=300, bbox_inches='tight', pad_inches=0.1)
    
    plt.show()

    # Return results in a DataFrame
    importance_df = pd.DataFrame({
        "Feature": feature_importance.index,
        "Importance Score": feature_importance.values
    })

    return importance_df

In [8]:
def pca_correlation_heatmap(data, num_components=5, png_file_name="pca_correlation_heatmap.png", figsize=(10, 6)):
    """
    Creates a heatmap showing the correlation between original features and principal components.

    Parameters:
    - data: DataFrame with input data for PCA (samples in rows, features in columns)
    - num_components: Number of principal components to analyze (default: 5)

    Returns:
    - DataFrame with correlation values between features and principal components
    """

    # Perform PCA
    pca = PCA(n_components=min(num_components, data.shape[1]))
    pca.fit(data)

    # Extract loadings (correlation-like coefficients)
    loadings = pd.DataFrame(pca.components_.T, index=data.columns, columns=[f"PC{i+1}" for i in range(num_components)])

    # Create correlation heatmap
    plt.figure(figsize=figsize)
    sns.heatmap(loadings, annot=True, cmap="coolwarm", center=0, linewidths=0.5, fmt=".2f")
    plt.title("PCA Feature-Component Correlation Heatmap", fontsize=14)
    plt.xlabel("Principal Components", fontsize=12)
    plt.ylabel("Original Features", fontsize=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)

    plt.savefig(png_file_name, dpi=300, bbox_inches='tight', pad_inches=0.1)
    
    plt.show()

    return loadings
